In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path

if "notebooks" in os.getcwd():
    os.chdir(Path.cwd().parent)

In [ ]:
import logging

logger = logging.getLogger("ts_visualization")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter("%(levelname)s: %(message)s")
    handler.setFormatter(formatter)
    logger.addHandler(handler)


In [ ]:
# from xaitimesynth.generators import TimeSeriesGenerator
# import numpy as np
# import pandas as pd
# import logging
# from lets_plot import *

# LetsPlot.setup_html()

# TimeSeriesGenerator()


# Helper functions

# Synth Data

In [ ]:
"""
Example usage of the XAITimeSynth package.

This example demonstrates how to create a synthetic time series dataset
with two classes, where one has discriminative features and the other doesn't.
"""

import numpy as np
from xaitimesynth import (
    TimeSeriesBuilder,
    random_walk,
    gaussian,
    shapelet,
    level_change,
    manual,
    peak,
)

# Create a dataset with two classes using the fluent API
dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=200, random_state=42)
    .for_class(0)  # Class 0: No discriminative features
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .for_class(1)  # Class 1: Has discriminative features
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(shapelet(scale=1.0), start_pct=0.4, end_pct=0.6)
    .add_feature(level_change(amplitude=0.5), start_pct=0.7, end_pct=0.9)
    .build(train_test_split=0.8)
)

# Print dataset structure
print("Dataset keys:", list(dataset.keys()))
print(f"X shape: {dataset['X'].shape}")
print(f"y shape: {dataset['y'].shape}")
print(f"Classes: {np.unique(dataset['y'])}")
print(f"Class distribution: {np.bincount(dataset['y'])}")


# Example with random feature locations
random_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=200, random_state=42)
    .for_class(0)  # Class 0: No discriminative features
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .for_class(1)  # Class 1: Has discriminative features with random locations
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(shapelet(scale=1.0), random_location=True, length_pct=0.2)
    .add_feature(level_change(amplitude=0.5), random_location=True, length_pct=0.2)
    .build()
)


# Create a custom component and use it
def my_seasonal_pattern(n_timesteps, rng, frequency=0.1, amplitude=1.0):
    """Generate a custom seasonal pattern."""
    t = np.arange(n_timesteps)
    return amplitude * np.sin(2 * np.pi * frequency * t / n_timesteps)


custom_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=10)
    .for_class(0)
    .add_signal(manual(generator=my_seasonal_pattern, frequency=0.05, amplitude=1.0))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .for_class(1)
    .add_signal(manual(generator=my_seasonal_pattern, frequency=0.05, amplitude=1.0))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(peak(amplitude=2.0, width=5), start_pct=0.4, end_pct=0.6)
    .build()
)


In [ ]:
# Create a dataset with the specified characteristics and higher frequency sine wave
sine_levelshift_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=200, random_state=42)
    .for_class(
        0
    )  # Class 0: Higher frequency sine wave pattern as base signal and level shift
    .add_signal(
        manual(
            generator=lambda n_timesteps, rng: np.sin(
                5 * np.linspace(0, 2 * np.pi, n_timesteps)
            )
        )
    )  # 5 cycles
    .add_feature(level_change(amplitude=2.0), start_pct=0.5, end_pct=0.7)
    .for_class(1)  # Class 1: Constant zero and level shift
    .add_signal(manual(generator=lambda n_timesteps, rng: np.zeros(n_timesteps)))
    .add_feature(level_change(amplitude=2.0), start_pct=0.5, end_pct=0.7)
    .build()
)


In [ ]:
random_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=50, random_state=42)
    # Class 0: No discriminative features
    .for_class(0)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    # Class 1: Random walk with randomly located features
    .for_class(1)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(shapelet(scale=1.2), random_location=True, length_pct=0.15)
    .add_feature(peak(amplitude=1.0, width=5), random_location=True, length_pct=0.1)
    .build()
)
print(random_dataset.keys())


random_dataset["X"].shape

In [ ]:
"""
Examples demonstrating the enhanced visualization options for XAITimeSynth
"""

import numpy as np
from lets_plot import *
from xaitimesynth import (
    TimeSeriesBuilder,
    random_walk,
    gaussian,
    # sine_wave,
    level_change,
    peak,
)
from xaitimesynth.visualization import (
    create_ts_visualization,
    plot_class_comparison,
)

# Enable lets_plot for notebook display
LetsPlot.setup_html()

# Create a dataset with three classes
dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=30, random_state=42)
    # Class 0: Random walk with noise only (baseline)
    .for_class(0)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    # Class 1: Random walk + sine wave feature + level change
    .for_class(1)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(sine_wave(frequency=0.2, amplitude=1.5), start_pct=0.3, end_pct=0.5)
    .add_feature(level_change(amplitude=0.7), start_pct=0.7, end_pct=0.9)
    # Class 2: Random walk + peak
    .for_class(2)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(peak(amplitude=2.0, width=5), start_pct=0.4, end_pct=0.6)
    .build()
)

# Get sample indices (first sample from each class)
sample_indices = {}
for class_label in np.unique(dataset["y"]):
    sample_indices[class_label] = np.where(dataset["y"] == class_label)[0][0]

# Example 1: Default visualization with ordered components
# Note: Default order is now "Full Series", "Features", "Base Structure", "Noise"
print("Default visualization with ordered components:")
default_plot = create_ts_visualization(
    dataset, sample_indices=sample_indices, panel_width=900, panel_height=500
)
default_plot.show()

# Example 2: Filtered components - only show Full Series and Features
print("\nFiltered components (Full Series and Features only):")
filtered_plot = create_ts_visualization(
    dataset,
    sample_indices=sample_indices,
    components=["Full Series", "Features"],  # Only show these components
    panel_width=900,
    panel_height=350,
)
filtered_plot.show()

# Example 3: Free Y scaling for better visibility of each component
print("\nFree Y-axis scaling:")
free_y_plot = create_ts_visualization(
    dataset,
    sample_indices=sample_indices,
    free_y=True,  # Use free y-scales
    panel_width=900,
    panel_height=500,
)
free_y_plot.show()

# Example 4: Single-row mode with class colors
print("\nSingle-row mode (all classes in one row with colors):")
single_row_plot = create_ts_visualization(
    dataset,
    sample_indices=sample_indices,
    components=["Full Series", "Features", "Base Structure"],
    free_y=True,  # Free y-scales often look better in single row mode
    panel_width=900,
    panel_height=300,
)
single_row_plot.show()

# Example 5: Class comparison alternatives
print("\nClass comparison - default (one panel per class):")
default_comparison = plot_class_comparison(dataset, panel_width=800, panel_height=300)
default_comparison.show()

print("\nClass comparison - single row with colors:")
single_row_comparison = plot_class_comparison(
    dataset,
    panel_width=600,
    panel_height=250,
)
single_row_comparison.show()

# Example 6: Custom ordering of components
print("\nCustom component ordering:")
custom_order_plot = create_ts_visualization(
    dataset,
    sample_indices=sample_indices,
    components=[
        "Features",
        "Full Series",
        "Noise",
    ],  # Custom order, excluding Base Structure
    panel_width=900,
    panel_height=400,
)
custom_order_plot.show()

print("Examples completed!")


In [ ]:
# Create a dataset with three classes
dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=30, random_state=42)
    # Class 0: Random walk with noise only (baseline)
    .for_class(0)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    # Class 1: Random walk + sine wave feature + level change
    .for_class(1)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(sine_wave(frequency=0.2, amplitude=1.5), start_pct=0.3, end_pct=0.5)
    .add_feature(level_change(amplitude=0.7), start_pct=0.7, end_pct=0.9)
    # Class 2: Random walk + peak
    .for_class(2)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(peak(amplitude=2.0, width=5), start_pct=0.4, end_pct=0.6)
    .build()
)
print(dataset.keys())


In [ ]:
custom_order_plot = create_ts_visualization(
    dataset,
    sample_indices=sample_indices,
    components=[
        "Complete Series",
        "Features",
        "Foundation",
        # "Noise",
    ],  # Custom order, excluding Base Structure
    panel_width=900,
    panel_height=400,
)
custom_order_plot.show()

In [ ]:
"""
Example demonstrating the fixed component ordering using as_discrete
"""

import numpy as np
from lets_plot import *
from xaitimesynth import (
    TimeSeriesBuilder,
    random_walk,
    gaussian,
    # sine_wave,
    level_change,
    peak,
)
from xaitimesynth.visualization import create_ts_visualization

# Enable lets_plot for notebook display
LetsPlot.setup_html()

# Create a dataset with three classes
dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=30, random_state=42)
    # Class 0: Random walk with noise only (baseline)
    .for_class(0)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    # Class 1: Random walk + sine wave feature + level change
    .for_class(1)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(sine_wave(frequency=0.2, amplitude=1.5), start_pct=0.3, end_pct=0.5)
    .add_feature(level_change(amplitude=0.7), start_pct=0.7, end_pct=0.9)
    # Class 2: Random walk + peak
    .for_class(2)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(peak(amplitude=2.0, width=5), start_pct=0.4, end_pct=0.6)
    .build()
)

# Get sample indices (first sample from each class)
sample_indices = {}
for class_label in np.unique(dataset["y"]):
    sample_indices[class_label] = np.where(dataset["y"] == class_label)[0][0]

# Example 1: Default component order (Full Series, Features, Base Structure, Noise)
print("Default component order:")
default_order_plot = create_ts_visualization(
    dataset, sample_indices=sample_indices, panel_width=225, panel_height=175
)
default_order_plot.show()

# Example 2: Custom component order
print("\nCustom component order:")
custom_order_plot = create_ts_visualization(
    dataset,
    sample_indices=sample_indices,
    components=["Full Series", "Features", "Noise", "Base Structure"],  # Custom order
    panel_width=225,
    panel_height=175,
)
custom_order_plot.show()

# Example 3: Reversed component order
print("\nReversed component order:")
reversed_order_plot = create_ts_visualization(
    dataset,
    sample_indices=sample_indices,
    components=[
        "Noise",
        "Base Structure",
        "Features",
        "Full Series",
    ],  # Completely reversed
    panel_width=225,
    panel_height=175,
)
reversed_order_plot.show()

# Example 4: Partial component selection with specific order
print("\nSelected components in specific order:")
partial_order_plot = create_ts_visualization(
    dataset,
    sample_indices=sample_indices,
    components=["Features", "Full Series"],  # Only these two in this order
    panel_width=300,
    panel_height=200,
)
partial_order_plot.show()

# Example 5: Transposed layout (components as rows) with correct ordering
print("\nTransposed layout with correct component ordering:")
transposed_plot = create_ts_visualization(
    dataset,
    sample_indices=sample_indices,
    components=["Full Series", "Features", "Base Structure", "Noise"],
    facet_order={"x": "class", "y": "component"},  # Transpose
    panel_width=250,
    panel_height=150,
)
transposed_plot.show()

print("Examples completed!")
